In [5]:
#import kagglehub

#kagglehub.login()

In [6]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("bielbauza/hwd-dataset",
#    output_dir="/home/jovyan/work/HWD+/Datasets")

#print("Path to dataset files:", path)

In [14]:
# Generic
import numpy as np
from numpy import *
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
# Set the seed
seed = 42

random.seed(seed)
np.random.seed(seed)

# Loading dataset

The dataset WriterInfo contains multiple variables that are of no use for use like gender, age... In this case we erase all variables apart from the ID and category

In [8]:
dataset = pd.DataFrame(
    np.load("/home/jovyan/work/HWD+/Datasets/WriterInfo.npy", allow_pickle=True)
)
print(dataset.shape)
print(dataset.head())
print(dataset.columns.tolist())

(13580, 9)
   0  1   2  3    4  5  6  7  8
0  1  0  29  1  173  0  0  2  1
1  1  0  29  1  173  0  0  2  1
2  1  0  29  1  173  0  0  2  1
3  1  0  29  1  173  0  0  2  1
4  1  0  29  1  173  0  0  2  1
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [9]:
labels = dataset.drop(columns=[1,2, 3, 4, 5, 6, 7, 8])
labels = labels.rename(columns={
    0: "label",
})

In [10]:
print(labels.shape)

(13580, 1)


# Creating final dataset

The images are ordered by the ID, as such we only need to combine both with a Class to ensure pytorch compatibility.

In [11]:
images=np.load(r"/home/jovyan/work/HWD+/Datasets/Images(500x500).npy")

In [12]:
labels["image"] = list(images)

In [15]:
class ImageDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

        assert len(images) == len(labels), \
            "Number of images and labels must match"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.as_tensor(
            self.images[idx], dtype=torch.float32
        )

        label = torch.as_tensor(
            self.labels[idx], dtype=torch.long
        )

        # Add channel dimension for grayscale images
        if image.ndim == 2:
            image = image.unsqueeze(0)

        # Convert 1 channel to 3 channels
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)

        return image, label

In [16]:
labels_array = labels["label"].to_numpy()
dataset = ImageDataset(images, labels_array)

We check how many of each category we have.

In [17]:
print("Unique labels:", labels["label"].unique())
print("Number of unique labels:", labels["label"].nunique())
print("Values of each category:",labels["label"].value_counts())

Unique labels: [1 2 3 4 5 6 7 8 9 0]
Number of unique labels: 10
Values of each category: label
1    1358
2    1358
3    1358
4    1358
5    1358
6    1358
7    1358
8    1358
9    1358
0    1358
Name: count, dtype: int64


The dataset is completely homegenous so we need not worry about unbalanced information. We now save the dataset for further use.

In [18]:
torch.save({
    'images': images,
    'labels': labels
}, 'train_dataset.pth')

AttributeError: 'ImageDataset' object has no attribute 'to_csv'